# codedet on Kaggle (manual fallback)

Use this when `scripts/kaggle_drive.py` cannot push a kernel from the laptop (for example when the
installed Kaggle CLI only accepts the legacy key for read operations). It runs exactly the same code.

**Notebook settings:** Accelerator **GPU T4 x1**, Internet **On**, and add the dataset
`daniilor/semeval-2026-task13` under *Input*.

Run `smoke` first (~5 min), then switch `PRESET` to `lite` (~40 min) and run again — finished stages
are cached by marker files, so a re-run only does what is missing.

In [ ]:
PRESET = "smoke"
REPO = "https://github.com/ranjankndev/codedetection-human-machine.git"
OUT = "/kaggle/working/results"

In [ ]:
import glob, torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True)[:10]:
    print(p)

In [ ]:
# Never reinstall torch/transformers on Kaggle - only what is missing.
!git clone -q $REPO /kaggle/working/codedet
%cd /kaggle/working/codedet
!pip install -q -e . 2>&1 | tail -1
for pkg in ("peft", "datasketch", "lightgbm"):
    try:
        __import__(pkg)
    except ImportError:
        !pip install -q {pkg} 2>&1 | tail -1

In [ ]:
!python -u scripts/run_all.py --preset $PRESET --data /kaggle/input --out $OUT

In [ ]:
import json
status = json.load(open(f"{OUT}/status.json"))
print(status["status"], status["elapsed_minutes"], "min")
for s in status["stages"]:
    print(f"{s['stage']:28s} {s['status']:8s} {s.get('seconds', 0):6.0f}s")
print(open(f"{OUT}/summary.md").read()[:4000])

In [ ]:
# Keep the output small: JSON / MD / PNG only (no parquet, npz or checkpoints)
import shutil
for d in ("data", "preds", "ckpt", "cache"):
    shutil.rmtree(f"{OUT}/{d}", ignore_errors=True)
!du -sh $OUT && find $OUT -type f | head -30